STEP 1. NSMC 데이터 분석 및 Huggingface dataset 구성
데이터셋은 깃허브(e9t/nsmc)에서 ratings_train.txt·ratings_test.txt 를 내려받거나, Huggingface datasets 의 e9t/nsmc에서 load_dataset("e9t/nsmc", revision="refs/convert/parquet") 로 가져올 수 있습니다(이 데이터셋은 로딩 스크립트 방식이라 datasets 4 이상에서는 parquet 변환 브랜치를 지정해야 합니다). 앞에서 배운 두 방법(불러오기·직접 가공)을 모두 써 보세요.

STEP 2. klue/bert-base model 및 tokenizer 불러오기

STEP 3. 위에서 불러온 tokenizer으로 데이터셋을 전처리하고, model 학습 진행해 보기

STEP 4. Fine-tuning을 통하여 모델 성능(accuracy) 향상시키기
데이터 전처리, TrainingArguments 등을 조정하여 모델의 정확도를 90% 이상으로 끌어올려봅시다.

STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교
아래 링크를 바탕으로 bucketing과 dynamic padding이 무엇인지 알아보고, 이들을 적용하여 model을 학습시킵니다.
Data Collator —DataCollatorWithPadding
TrainingArguments 의train_sampling_strategy="group_by_length"
 (transformers v5 부터 group_by_length=True 대신 이 설정을 씁니다)

STEP 4에 학습한 결과와 bucketing을 적용하여 학습시킨 결과를 비교해보고, 모델 성능 향상과 훈련 시간 두 가지 측면에서 각각 어떤 이점이 있는지 비교해봅시다.



In [1]:
import torch
import numpy
import transformers
import datasets
import os
import evaluate
from datasets import load_dataset

print(f"transformers 버전 : {transformers.__version__}")
print(f"datasets     버전 : {datasets.__version__}")
print(f"evaluate     버전 : {evaluate.__version__}")
print(f"torch        버전 : {torch.__version__}")

transformers 버전 : 5.17.0
datasets     버전 : 5.0.1
evaluate     버전 : 0.4.6
torch        버전 : 2.5.1+cu121


In [2]:
# 1. 로컬 저장 디렉토리 설정 및 생성
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

print("데이터셋 다운로드 중...")
# Hugging Face Hub에서 NSMC 데이터셋 로드 (parquet 브랜치 지정)
dataset = load_dataset("e9t/nsmc", revision="refs/convert/parquet")

print("\n--- 불러온 데이터셋 구조 ---")
print(dataset)

# 2. ./data/ 폴더에 로컬 저장
save_path = os.path.join(DATA_DIR, "nsmc_dataset")
dataset.save_to_disk(save_path)

print(f"\n데이터셋이 성공적으로 '{save_path}' 폴더에 저장되었습니다!")

데이터셋 다운로드 중...



--- 불러온 데이터셋 구조 ---
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/150000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]


데이터셋이 성공적으로 './data\nsmc_dataset' 폴더에 저장되었습니다!


In [3]:
from datasets import load_from_disk

# 로컬 폴더에서 데이터셋 불러오기
local_dataset = load_from_disk("./data/nsmc_dataset")

# 데이터 샘플 확인 (train 데이터 첫 번째 항목)
print("--- 로컬 데이터 샘플 확인 ---")
print("Train 개수:", len(local_dataset["train"]))
print("첫 번째 데이터:", local_dataset["train"][0])

--- 로컬 데이터 샘플 확인 ---
Train 개수: 150000
첫 번째 데이터: {'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}


In [4]:
import os
from datasets import load_from_disk
from transformers import AutoTokenizer

# 1. 로컬 데이터셋 및 토크나이저 불러오기
DATA_PATH = "./data/nsmc_dataset"
MODEL_NAME = "klue/bert-base"

print("로컬 데이터셋 로드 중...")
dataset = load_from_disk(DATA_PATH)

print(f"'{MODEL_NAME}' 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# 2. 전처리(Tokenization) 함수 정의
def preprocess_function(examples):
    # NSMC 텍스트 컬럼 이름은 'document'입니다.
    # truncation=True: max_length(128) 넘는 문장은 자름
    return tokenizer(
        examples["document"],
        truncation=True,
        max_length=128,
    )


# 3. dataset.map()으로 전체 데이터 배치 전처리
print("전처리(Tokenization) 진행 중...")
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["id", "document"],  # 모델 학습에 불필요한 원본 텍스트/ID 제거
)

print("\n--- 전처리 완료된 데이터 구조 ---")
print(tokenized_datasets)
print("\n첫 번째 샘플 키 목록:", tokenized_datasets["train"][0].keys())

# 4. 전처리 완료된 데이터셋 로컬 저장
PROCESSED_DATA_PATH = "./data/nsmc_tokenized"
tokenized_datasets.save_to_disk(PROCESSED_DATA_PATH)
print(f"\n전처리 완료된 데이터가 '{PROCESSED_DATA_PATH}'에 저장되었습니다!")

로컬 데이터셋 로드 중...
'klue/bert-base' 토크나이저 로드 중...
전처리(Tokenization) 진행 중...

--- 전처리 완료된 데이터 구조 ---
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

첫 번째 샘플 키 목록: dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


Saving the dataset (0/1 shards):   0%|          | 0/150000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]


전처리 완료된 데이터가 './data/nsmc_tokenized'에 저장되었습니다!


In [5]:
from datasets import load_from_disk
from transformers import AutoTokenizer

# 1. 전처리된 데이터와 토크나이저 불러오기
PROCESSED_DATA_PATH = "./data/nsmc_tokenized"
MODEL_NAME = "klue/bert-base"

tokenized_dataset = load_from_disk(PROCESSED_DATA_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. 첫 번째 데이터 샘플 추출 (train 데이터 기준)
sample = tokenized_dataset["train"][0]

print("=== 1. 저장된 딕셔너리 Key 목록 ===")
print(sample.keys())

print("\n=== 2. input_ids (토큰 ID 숫자 변환값) ===")
print(sample["input_ids"][:15])  # 처음 15개만 출력

print("\n=== 3. attention_mask (패딩 구분용 마스크) ===")
print(sample["attention_mask"][:15])

print("\n=== 4. label (정답 라벨: 0=부정, 1=긍정) ===")
print(sample["label"])

print("\n=== 5. input_ids를 실제 텍스트로 역변환(Decode) ===")
decoded_text = tokenizer.decode(sample["input_ids"])
print(decoded_text)

=== 1. 저장된 딕셔너리 Key 목록 ===
dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])

=== 2. input_ids (토큰 ID 숫자 변환값) ===
[2, 1376, 831, 2604, 18, 18, 4229, 9801, 2075, 2203, 2182, 4243, 3]

=== 3. attention_mask (패딩 구분용 마스크) ===
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

=== 4. label (정답 라벨: 0=부정, 1=긍정) ===
0

=== 5. input_ids를 실제 텍스트로 역변환(Decode) ===
[CLS] 아 더빙.. 진짜 짜증나네요 목소리 [SEP]


In [7]:
import os
import numpy as np
import evaluate
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# 1. 로컬 데이터 및 토크나이저 불러오기
PROCESSED_DATA_PATH = "./data/nsmc_tokenized"
MODEL_NAME = "klue/bert-base"

print("전처리된 데이터셋 및 토크나이저 로드 중...")
tokenized_datasets = load_from_disk(PROCESSED_DATA_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. 동적 패딩용 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3. 모델 초기화 (긍정/부정 이진 분류 num_labels=2)
print(f"'{MODEL_NAME}' v5 규격 모델 초기화 중...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 4. 평가 지표 (Accuracy) 함수
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# 5. TrainingArguments 세팅 (transformers v5 규격)
training_args = TrainingArguments(
    output_dir="./results_step4",
    learning_rate=2e-5,
    per_device_train_batch_size=8,      
    gradient_accumulation_steps=4,      
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",            # v5 표준 평가 주기 설정
    save_strategy="epoch",
    load_best_model_at_end=True,      # 최고 성능 모델 자동 로드
    metric_for_best_model="accuracy",
    report_to="none"
)

# 6. Trainer 인스턴스화 (v5 표준 processing_class 인자 사용)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,       # v5 최신 표준
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7. 실제 파인튜닝 시작
print("\n🚀 [v5.17.0] STEP 3 & 4: 파인튜닝을 시작합니다...")
trainer.train()

# 8. 최종 결과 확인 및 저장
print("\n=== 최종 평가 결과 ===")
eval_results = trainer.evaluate()
print(f"Test Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")

# 저장
FINAL_MODEL_PATH = "./models/nsmc_bert_base"
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"모델과 토크나이저 저장 완료: {FINAL_MODEL_PATH}")

전처리된 데이터셋 및 토크나이저 로드 중...
'klue/bert-base' v5 규격 모델 초기화 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 [v5.17.0] STEP 3 & 4: 파인튜닝을 시작합니다...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.007150,0.240485,0.900200
2,0.770875,0.238225,0.907320
3,0.501407,0.306090,0.906360


...


=== 최종 평가 결과 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.501407,0.238225,3,0.907320


Test Accuracy: 90.73%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델과 토크나이저 저장 완료: ./models/nsmc_bert_base


In [ ]:
# import torch
# import gc

# gc.collect()
# torch.cuda.empty_cache()

In [9]:
import os
import time
import numpy as np
import evaluate
import torch
import gc
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# 0. 이전 GPU 메모리 청소
gc.collect()
torch.cuda.empty_cache()

# 1. 데이터 및 토크나이저 불러오기
PROCESSED_DATA_PATH = "./data/nsmc_tokenized"
MODEL_NAME = "klue/bert-base"

print("전처리된 데이터셋 및 토크나이저 로드 중...")
tokenized_datasets = load_from_disk(PROCESSED_DATA_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. 동적 패딩 Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3. 모델 초기화
print(f"'{MODEL_NAME}' 모델 초기화 중...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 4. 평가 지표 (Accuracy) 함수
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# 5. TrainingArguments 세팅 (Bucketing 적용)
training_args = TrainingArguments(
    output_dir="./results_step5_bucketing",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=0.1,
    
    # 🌟 STEP 5 핵심: Bucketing 옵션 활성화
    train_sampling_strategy="group_by_length",            # 문장 길이에 따라 데이터 묶기
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

# 6. Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 7. 학습 시작 (시간 측정)
print("\n🚀 [STEP 5] Bucketing 적용 파인튜닝을 시작합니다...")
start_time = time.time()
trainer.train()
end_time = time.time()

elapsed_time = end_time - start_time
print(f"\n⏱️ 버케팅 적용 학습 완료 소요 시간: {elapsed_time / 60:.2f}분")

# 8. 최종 평가 및 저장
print("\n=== 최종 평가 결과 ===")
eval_results = trainer.evaluate()
print(f"Test Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")

FINAL_MODEL_PATH = "./models/nsmc_bert_bucketing"
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"모델 저장 완료: {FINAL_MODEL_PATH}")

전처리된 데이터셋 및 토크나이저 로드 중...
'klue/bert-base' 모델 초기화 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 [STEP 5] Bucketing 적용 파인튜닝을 시작합니다...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.023814,0.248859,0.900060
2,0.747260,0.243641,0.906880
3,0.529852,0.304762,0.906140


...


⏱️ 버케팅 적용 학습 완료 소요 시간: 38.53분

=== 최종 평가 결과 ===


Training Loss,Validation Loss,Epoch,Accuracy
0.529852,0.243641,3,0.906880


Test Accuracy: 90.69%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료: ./models/nsmc_bert_bucketing


In [10]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 저장된 모델 및 토크나이저 로드
MODEL_PATH = "./models/nsmc_bert_bucketing"

print(f"'{MODEL_PATH}'에서 저장된 모델 및 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Evaluation 모드 전환 및 GPU/CPU 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. 감정 분석(긍정/부정) 예측 함수 정의
def predict_sentiment(text):
    # 입력 문장 토큰화
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(device)

    # 추론 (기억 자원 절약을 위해 gradient 계산 비활성화)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Softmax를 사용하여 확률(Probability)값 계산
        probs = F.softmax(logits, dim=-1).squeeze().tolist()

    neg_prob, pos_prob = probs[0], probs[1]
    
    # 더 높은 확률의 라벨 결정 (0: 부정, 1: 긍정)
    label = "긍정 (Positive) 😊" if pos_prob > neg_prob else "부정 (Negative) 😢"
    confidence = max(pos_prob, neg_prob) * 100

    print(f"\n💬 리뷰: \"{text}\"")
    print(f"📊 예측: {label} (신뢰도: {confidence:.2f}%)")
    print(f"   - 긍정 확률: {pos_prob * 100:.2f}%")
    print(f"   - 부정 확률: {neg_prob * 100:.2f}%")


# 3. 테스트할 영화 리뷰 문장들
test_reviews = [
    "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!",
    "시간 아까우니까 절대 보지 마세요. 돈 날렸음.",
    "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요.",
    "전개도 어색하고 결말도 허무함. 왜 평점이 높은지 이해가 안 가네."
]

# 4. 추론 실행
print("\n🚀 [NSMC 감정 분석 추론 시작]")
for review in test_reviews:
    predict_sentiment(review)

'./models/nsmc_bert_bucketing'에서 저장된 모델 및 토크나이저 로드 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 [NSMC 감정 분석 추론 시작]

💬 리뷰: "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!"
📊 예측: 긍정 (Positive) 😊 (신뢰도: 99.62%)
   - 긍정 확률: 99.62%
   - 부정 확률: 0.38%

💬 리뷰: "시간 아까우니까 절대 보지 마세요. 돈 날렸음."
📊 예측: 부정 (Negative) 😢 (신뢰도: 99.89%)
   - 긍정 확률: 0.11%
   - 부정 확률: 99.89%

💬 리뷰: "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요."
📊 예측: 긍정 (Positive) 😊 (신뢰도: 99.77%)
   - 긍정 확률: 99.77%
   - 부정 확률: 0.23%

💬 리뷰: "전개도 어색하고 결말도 허무함. 왜 평점이 높은지 이해가 안 가네."
📊 예측: 부정 (Negative) 😢 (신뢰도: 99.87%)
   - 긍정 확률: 0.13%
   - 부정 확률: 99.87%


In [11]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 🌟 기존 Baseline 모델 경로로 설정
MODEL_PATH = "./models/nsmc_bert_base"

print(f"'{MODEL_PATH}'에서 기존 모델 및 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().tolist()

    neg_prob, pos_prob = probs[0], probs[1]
    label = "긍정 (Positive) 😊" if pos_prob > neg_prob else "부정 (Negative) 😢"
    confidence = max(pos_prob, neg_prob) * 100

    print(f"\n💬 리뷰: \"{text}\"")
    print(f"📊 예측: {label} (신뢰도: {confidence:.2f}%)")
    print(f"   - 긍정 확률: {pos_prob * 100:.2f}%")
    print(f"   - 부정 확률: {neg_prob * 100:.2f}%")

test_reviews = [
    "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!",
    "시간 아까우니까 절대 보지 마세요. 돈 날렸음.",
    "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요.",
    "전개도 어색하고 결말도 허무함. 왜 평점이 높은지 이해가 안 가네."
]

print("\n🚀 [Baseline 모델 감정 분석 추론 시작]")
for review in test_reviews:
    predict_sentiment(review)

'./models/nsmc_bert_base'에서 기존 모델 및 토크나이저 로드 중...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 [Baseline 모델 감정 분석 추론 시작]

💬 리뷰: "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!"
📊 예측: 긍정 (Positive) 😊 (신뢰도: 99.35%)
   - 긍정 확률: 99.35%
   - 부정 확률: 0.65%

💬 리뷰: "시간 아까우니까 절대 보지 마세요. 돈 날렸음."
📊 예측: 부정 (Negative) 😢 (신뢰도: 99.87%)
   - 긍정 확률: 0.13%
   - 부정 확률: 99.87%

💬 리뷰: "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요."
📊 예측: 긍정 (Positive) 😊 (신뢰도: 99.65%)
   - 긍정 확률: 99.65%
   - 부정 확률: 0.35%

💬 리뷰: "전개도 어색하고 결말도 허무함. 왜 평점이 높은지 이해가 안 가네."
📊 예측: 부정 (Negative) 😢 (신뢰도: 99.85%)
   - 긍정 확률: 0.15%
   - 부정 확률: 99.85%


In [12]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 두 모델 경로 설정
BASE_MODEL_PATH = "./models/nsmc_bert_base"
BUCKET_MODEL_PATH = "./models/nsmc_bert_bucketing"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# 2. Baseline 모델 로드
print(f"\n1) Baseline 모델 로드 중 ({BASE_MODEL_PATH})...")
tokenizer_base = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
model_base = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_PATH).to(device)
model_base.eval()

# 3. Bucketing 모델 로드
print(f"2) Bucketing 모델 로드 중 ({BUCKET_MODEL_PATH})...")
tokenizer_bucket = AutoTokenizer.from_pretrained(BUCKET_MODEL_PATH)
model_bucket = AutoModelForSequenceClassification.from_pretrained(BUCKET_MODEL_PATH).to(device)
model_bucket.eval()

# 4. 개별 모델 예측 함수
def get_prediction(model, tokenizer, text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().tolist()

    neg_prob, pos_prob = probs[0], probs[1]
    label = "긍정 😊" if pos_prob > neg_prob else "부정 😢"
    return label, pos_prob * 100, neg_prob * 100

# 5. 두 모델 비교 출력 함수
def compare_sentence(text):
    label_base, pos_base, neg_base = get_prediction(model_base, tokenizer_base, text)
    label_bucket, pos_bucket, neg_bucket = get_prediction(model_bucket, tokenizer_bucket, text)

    print(f"\n💬 리뷰: \"{text}\"")
    print(f"┌──────────────────────┬──────────────────────────────────────────┐")
    print(f"│ 모델 구분            │ 예측 결과 (긍정 확률 / 부정 확률)        │")
    print(f"├──────────────────────┼──────────────────────────────────────────┤")
    print(f"│ Baseline (58분)      │ {label_base:<8} (긍정: {pos_base:5.2f}% / 부정: {neg_base:5.2f}%) │")
    print(f"│ Bucketing (38분)     │ {label_bucket:<8} (긍정: {pos_bucket:5.2f}% / 부정: {neg_bucket:5.2f}%) │")
    print(f"└──────────────────────┴──────────────────────────────────────────┘")

# 6. 다양한 테스트 문장 모음
test_reviews = [
    "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!",
    "시간 아까우니까 절대 보지 마세요. 돈 날렸음.",
    "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요.",
    "전개도 어색하고 결말도 허무함. 왜 평점이 높은지 이해가 안 가네.",
    "소소하게 웃으면서 보기 좋은 영화. 큰 기대 안 하고 보면 재밌음."
]

# 7. 실행
print("\n🚀 [Baseline vs Bucketing 모델 예측 결과 비교]")
for review in test_reviews:
    compare_sentence(review)

사용 장치: cuda

1) Baseline 모델 로드 중 (./models/nsmc_bert_base)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2) Bucketing 모델 로드 중 (./models/nsmc_bert_bucketing)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 [Baseline vs Bucketing 모델 예측 결과 비교]

💬 리뷰: "이 영화 진짜 인생작입니다... 배우 연기부터 연출까지 완벽했어요!"
┌──────────────────────┬──────────────────────────────────────────┐
│ 모델 구분            │ 예측 결과 (긍정 확률 / 부정 확률)        │
├──────────────────────┼──────────────────────────────────────────┤
│ Baseline (58분)      │ 긍정 😊     (긍정: 99.35% / 부정:  0.65%) │
│ Bucketing (38분)     │ 긍정 😊     (긍정: 99.62% / 부정:  0.38%) │
└──────────────────────┴──────────────────────────────────────────┘

💬 리뷰: "시간 아까우니까 절대 보지 마세요. 돈 날렸음."
┌──────────────────────┬──────────────────────────────────────────┐
│ 모델 구분            │ 예측 결과 (긍정 확률 / 부정 확률)        │
├──────────────────────┼──────────────────────────────────────────┤
│ Baseline (58분)      │ 부정 😢     (긍정:  0.13% / 부정: 99.87%) │
│ Bucketing (38분)     │ 부정 😢     (긍정:  0.11% / 부정: 99.89%) │
└──────────────────────┴──────────────────────────────────────────┘

💬 리뷰: "처음엔 좀 지루했는데 뒤로 갈수록 몰입감이 대단하네요."
┌──────────────────────┬──────────────────────────────────────────┐
│ 모델 구분       

비교 항목,Baseline (일반 패딩),Bucketing (길이별 그룹화),변화 및 Trade-off 여부
연산 속도 (학습 시간),58분 13초,38분 31초,약 20분 (34%) 대폭 단축 ⚡
모델 성능 (Test Accuracy),90.73%,90.69%,0.04% 차이로 사실상 완전 동일 🎯

네, 맞습니다! 바로 그 **"고정으로 먹고 들어가는 메모리"** 존재 때문에 헷갈리셨던 겁니다.

모델을 학습시키려면 배치 사이즈에 상관없이 **기본적으로 GPU 메모리에 항상 올려두어야 하는 필수 요소들**이 있습니다.

---

### 고정 메모리를 구성하는 3가지 (배치 사이즈 = 1이어도 필수)

1. **모델 가중치 (Model Weights)**
* 모델의 파라미터(매개변수) 자체입니다.
* 예: 1억 개의 파라미터를 가진 FP32(32비트) 모델은 기본으로 **0.4 GB**를 차지합니다. 7B(70억 개) 모델은 기본 **14 GB** 이상을 차지합니다.


2. **옵티마이저 상태 (Optimizer States - AdamW 등)**
* **이게 메모리를 엄청나게 잡아먹는 주범입니다.**
* 가장 많이 쓰는 AdamW 옵티마이저는 학습 속도와 안정성을 위해 **모델 가중치 크기의 2배(Momentum, Variance)** 데이터값을 계속 GPU에 지니고 있어야 합니다.
* 즉, 모델 가중치에 들어가는 메모리의 **2~3배가 옵티마이저용 고정 메모리**로 추가로 쓰입니다.


3. **기울기 버퍼 (Gradient Buffers)**
* 가중치를 업데이트하기 위해 계산 결과를 담아둘 공간으로, **모델 가중치 크기만큼 1:1로 고정 할당**됩니다.



---

### 쉽게 비유하자면?

> **"트럭(GPU)에 짐(데이터)을 싣는 상황"**

* **트럭 자체의 무게 + 운전사 + 기름 (고정 메모리)**: 짐을 1개를 싣든 10개를 싣든 기본으로 발생하는 무게입니다. (예: 4.5톤)
* **짐의 무게 (가변 메모리 - 배치 사이즈)**: 배치 사이즈를 1개, 2개, 8개 늘릴 때마다 추가되는 순수 짐의 무게입니다. (예: 개당 0.3톤)

---

### 그래서 실제 메모리는 이렇게 늘어납니다

* **배치 1개**: 4.5GB (고정) + 0.3GB (배치1) = **4.8 GB**
* **배치 2개**: 4.5GB (고정) + 0.6GB (배치2) = **5.1 GB**
* **배치 4개**: 4.5GB (고정) + 1.2GB (배치4) = **5.7 GB**
* **배치 8개**: 4.5GB (고정) + 2.4GB (배치8) = **6.9 GB**

배치 사이즈를 1에서 8로 **8배** 늘려도, 실제 총 메모리는 **4.8 GB에서 6.9 GB로 약 40% 정도만 증가**하는 이유가 바로 이 **고정 메모리(4.5GB)** 때문입니다!

**결론부터 말씀드리면, 가능하고 실제로 NLP 분야에서 자주 사용하는 테크닉입니다!**

이를 구현하는 방법은 크게 두 가지 방식(패키징 방식 vs. 문서 스라이딩 방식)으로 나뉩니다.

---

### 1. 롱 텍스트 패키징 (Long Text Packaging / Chunking)

문장 단위로 자르는 것이 아니라, 전체 글을 길게 이어서 토큰화(Tokenize)한 뒤 **`max_len` 크기(예: 512 토큰)로 딱딱 잘라서 여러 개의 데이터**로 만드는 방식입니다.

* **동작 방식**:
1. 전체 텍스트(여러 문장)를 구분자(`[SEP]`나 `\n`)와 함께 하나의 긴 문자열로 합칩니다.
2. 전체를 토큰으로 변환한 뒤, 앞에서부터 `max_len` 개수(예: 512개)씩 잘라 하나의 샘플로 저장합니다.
3. 자르고 남은 뒷부분은 **버리지 않고 다음 데이터 샘플**이 됩니다.


* **장점**:
* 버려지는 데이터가 전혀 없어서 텍스트 정보를 100% 활용할 수 있습니다.
* Padding 토큰을 채울 필요가 없어 연산 효율이 극대화됩니다.


* **활용 사례**:
* GPT 계열의 사전 학습(Pre-training)이나 긴 문서 요약/학습 데이터를 만들 때 표준으로 사용됩니다.



---

### 2. 슬라이딩 윈도우 (Sliding Window / Overlapping Chunk)

문장을 자를 때 딱 맞춰서 잘라버리면, 문맥(Context)이 잘리는 문제점이 생깁니다. 이를 방지하기 위해 **일정 부분(Stride)을 겹치게(Overlap) 하면서 쪼개는 방식**입니다.

* **동작 방식**:
* 1번 조각: 0 ~ 512 토큰
* 2번 조각: 384 ~ 896 토큰 (앞 조각의 뒤쪽 128 토큰과 겹침)
* 3번 조각: 768 ~ 1280 토큰 (앞 조각의 뒤쪽 128 토큰과 겹침)


* **장점**:
* 앞뒤 문맥 정보가 끊기지 않고 자연스럽게 이어져 모델의 이해력이 높아집니다.


* **활용 사례**:
* 긴 문서 질문 답변(RAG / Question Answering) 시스템 구축 시 자주 사용합니다.



---

### Hugging Face Datasets로 간단히 구현하는 개념

Hugging Face의 `map` 함수를 쓸 때 `return_overflowing_tokens=True` 옵션을 주면, 긴 문장이 잘렸을 때 버려지지 않고 자동으로 여러 개로 분할되어 데이터셋에 추가됩니다.

```python
def chunk_text(examples):
    return tokenizer(
        examples["text"],
        max_length=512,
        truncation=True,
        return_overflowing_tokens=True,  # 잘린 뒷부분을 버리지 않고 새로운 데이터로 만듦
        stride=128,                       # (선택) 128 토큰씩 겹치게 잘라 문맥 유지
    )

```

---

### 요약

* 단순히 `truncation=True`만 하면 `max_len` 뒤는 삭제(버림)됩니다.
* 그러나 **Chunking / Sliding Window** 기법을 쓰면 잘린 뒷부분도 **독립된 다음 데이터로 재활용**하여 문장과 문맥을 모두 살릴 수 있습니다.